# 14 - Read permissions and service quota

Check the current account's control-plane actions and Southeast Asia quota for Azure ML and Container Apps. This notebook creates no resources or role assignments. Subscription IDs are used inside the script and are not saved in the report. Registering in 12 does not block this read-only step.


In [1]:
from pathlib import Path
import json
import platform
import subprocess
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())

cost = json.loads((ROOT / "reports/azure-cost-plan.json").read_text())
assert cost["within_planning_budget"], "Review the demo budget first."
print("Read-only checks; the cost plan is still an estimate.")


Read-only checks; the cost plan is still an estimate.


## Read access and quota

Run the cell, then save. Check checks_failed, control_plane_actions, training_quota_available, and container_apps_environment_slot_available. Missing permission or unknown quota is not a pass. The final CPU quota for a Container Apps environment is not available before the environment exists.


In [2]:
result = subprocess.run([sys.executable, "-m", "cloudlayer.access_check"],
                        cwd=ROOT, capture_output=True, text=True)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Access check failed. Read the traceback above.")
report = json.loads(result.stdout)
print(json.dumps(report, indent=2))
assert not report["checks_failed"], "Stop and send the report."


{
  "checked_at": "2026-10-11T13:59:57.042744+00:00",
  "region": "southeastasia",
  "checks_failed": [],
  "resources_created": false,
  "deploy_ready": false,
  "control_plane_actions": {
    "Microsoft.Resources/subscriptions/resourceGroups/write": true,
    "Microsoft.Storage/storageAccounts/write": true,
    "Microsoft.ContainerRegistry/registries/write": true,
    "Microsoft.App/managedEnvironments/write": true,
    "Microsoft.App/containerApps/write": true,
    "Microsoft.MachineLearningServices/workspaces/write": true,
    "Microsoft.OperationalInsights/workspaces/write": true,
    "Microsoft.Insights/components/write": true,
    "Microsoft.KeyVault/vaults/write": true,
    "Microsoft.Authorization/roleAssignments/write": true
  },
  "azure_ml_quota": [
    {
      "name": "TotalClusters",
      "description": "Total Clusters",
      "limit": 200,
      "current": 0,
      "remaining": 200
    },
    {
      "name": "TotalDedicatedCores",
      "description": "Total Cluster Ded

## Output

reports/azure-access-quota.json. Send the saved result. deploy_ready stays false because policies, data access, final app CPU quota, and unpriced costs remain open. This is not a setup error. Next: prepare the small Azure resource group and services.

[Azure ML quota API](https://learn.microsoft.com/en-us/rest/api/azureml/usages/list?view=rest-azureml-2026-03-01) | [Container Apps quota API](https://learn.microsoft.com/en-us/rest/api/resource-manager/containerapps/usages/list?view=rest-resource-manager-containerapps-2025-07-01)
